# Reviewer threshold and adaptive-onset runs

Threshold calibration and adaptive onset are separate resumable stages so they can follow the reviewer-recommended order. Preview is the default; both launch commands always contain `--resume`.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import os, shlex, subprocess, sys
def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'calibration_onset.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
RUNNER_ENV['MPLBACKEND'] = 'Agg'
RUNNER_ENV['PYTHONUNBUFFERED'] = '1'
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))
sys.path.insert(0, SOURCE_ROOT)
from calcium_transient_rising_flank.checkpointing import format_progress
RUN_THRESHOLD = False
RUN_ONSET = False
SEEDS = '1,2,3,4,5,6,7,8'
N_SURROGATES = 1000
OUTPUT_ROOT = PACKAGE_ROOT / 'outputs/revision_campaign'
def launch(command, enabled):
    print(f'[notebook] command: {shlex.join(command)}', flush=True)
    if enabled:
        print(format_progress(0, 1, label='Notebook stage') + ' | running', flush=True)
        subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)
        print(format_progress(1, 1, label='Notebook stage') + ' | complete', flush=True)
    else:
        print('[notebook] preview only; enable the RUN_* toggle to execute', flush=True)

In [ ]:
threshold_command = [
    RUNNER_PYTHON, 'examples/calibration_onset.py',
    '--components', 'threshold', '--seeds', SEEDS,
    '--n-estimator-surrogates', str(N_SURROGATES),
    '--output-dir', str(OUTPUT_ROOT / 'threshold_calibration'), '--resume',
]
launch(threshold_command, RUN_THRESHOLD)

In [ ]:
onset_command = [
    RUNNER_PYTHON, 'examples/calibration_onset.py',
    '--components', 'onset', '--seeds', SEEDS,
    '--output-dir', str(OUTPUT_ROOT / 'adaptive_onset'), '--resume',
]
launch(onset_command, RUN_ONSET)